In [43]:

import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import numpy as np
import pandas as pd
import matplotlib

import matplotlib.pyplot as plt

from config import (
    OHLCV_DIR, ASSET_DAILY_PATH, ASSETS_PATH,
    DATE_COL, TIMESTAMP_COL, CLOSE_COL, ASSET_ID_COL, MARKET_SYMBOL_COL,
)
from helperfunctions import HelperFunctions
helper = HelperFunctions()

print("Imports OK")

Imports OK


In [44]:
# --- Daten laden & vorverarbeiten ---
from helperfunctions import STABLECOIN_SYMBOLS

# Assets-Tabelle: asset_id <-> symbol
assets_df  = pd.read_parquet(ASSETS_PATH)[["asset_id", "symbol"]].drop_duplicates()
id_to_symbol = dict(zip(assets_df["asset_id"], assets_df["symbol"]))
sym_to_id    = dict(zip(assets_df["symbol"], assets_df["asset_id"]))

# Stablecoin asset_ids berechnen — werden aus der Point Cloud ausgeschlossen
excluded_ids: set[int] = set(
    assets_df.loc[assets_df["symbol"].isin(STABLECOIN_SYMBOLS), "asset_id"].tolist()
)
excluded_syms = sorted(assets_df.loc[assets_df["asset_id"].isin(excluded_ids), "symbol"])
print(f"Ausgeschlossene Stablecoins: {excluded_syms}")

ohlcv_files = sorted(OHLCV_DIR.glob("*.parquet"))
print(f"\nOHLCV-Dateien gefunden: {len(ohlcv_files)}")

raw = pd.concat([pd.read_parquet(f) for f in ohlcv_files], ignore_index=True)
raw["base_symbol"] = raw[MARKET_SYMBOL_COL].str.replace("USDT$", "", regex=True)
raw = raw.merge(assets_df.rename(columns={"symbol": "base_symbol"}), on="base_symbol", how="left")
raw = raw.dropna(subset=["asset_id"])
raw["asset_id"] = raw["asset_id"].astype(int)
raw[TIMESTAMP_COL] = pd.to_datetime(raw[TIMESTAMP_COL], utc=True)
raw[DATE_COL]      = raw[TIMESTAMP_COL].dt.normalize()

ohlcv_df = raw[[TIMESTAMP_COL, DATE_COL, ASSET_ID_COL, CLOSE_COL]].copy()

print(f"OHLCV geladen: {ohlcv_df.shape[0]:,} Zeilen | Assets: {ohlcv_df[ASSET_ID_COL].nunique()}")
print(f"Zeitraum: {ohlcv_df[DATE_COL].min().date()} → {ohlcv_df[DATE_COL].max().date()}")

universe_df = pd.read_parquet(ASSET_DAILY_PATH)
universe_df[DATE_COL] = pd.to_datetime(universe_df[DATE_COL], utc=True)
print(f"\nUniverse geladen: {universe_df.shape[0]:,} Zeilen")

Ausgeschlossene Stablecoins: ['DAI', 'PAXG', 'PYUSD', 'USD1', 'USDC', 'USDE', 'USDT', 'XAUT']

OHLCV-Dateien gefunden: 3
OHLCV geladen: 1,038,360 Zeilen | Assets: 41
Zeitraum: 2026-01-01 → 2026-03-31

Universe geladen: 4,013 Zeilen


In [45]:
# --- Point Cloud erstellen ---
END_DATE    = pd.Timestamp(ohlcv_df[DATE_COL].max().date(), tz="UTC")
WINDOW_DAYS = 30

result = helper.build_point_cloud(
    ohlcv_df         = ohlcv_df,
    universe_df      = universe_df,
    end_date         = END_DATE,
    window_size_days = WINDOW_DAYS,
    exclude_asset_ids= excluded_ids,
)

if result is None:
    print("Keine Daten für dieses Fenster — anderes end_date wählen.")
else:
    pc   = result["point_cloud"]
    aids = result["asset_ids"]
    syms = [id_to_symbol.get(i, f"ID:{i}") for i in aids]

    print("=" * 55)
    print(f"Fenster:      {result['start_date'].date()} → {result['end_date'].date()}")
    print(f"Punkte (w):   {result['n_points']}   ← Anzahl 5min-Kerzen")
    print(f"Dimensionen:  {result['n_dimensions']}  ← Anzahl Assets (ohne Stablecoins)")
    print(f"Point-Cloud-Shape: {pc.shape}")
    print(f"Assets: {syms}")
    print("=" * 55)
    df_preview = pd.DataFrame(pc[:5], columns=syms, index=result["timestamps"][:5])
    df_preview.index.name = "timestamp"
    display(df_preview.style.format("{:+.4%}"))

Fenster:      2026-03-02 → 2026-03-31
Punkte (w):   8639   ← Anzahl 5min-Kerzen
Dimensionen:  35  ← Anzahl Assets (ohne Stablecoins)
Point-Cloud-Shape: (8639, 35)
Assets: ['BTC', 'ETH', 'BNB', 'XRP', 'SOL', 'TRX', 'DOGE', 'ADA', 'BCH', 'LINK', 'ZEC', 'XLM', 'LTC', 'AVAX', 'SUI', 'TON', 'SHIB', 'WLFI', 'TAO', 'DOT', 'UNI', 'NEAR', 'ASTER', 'PEPE', 'AAVE', 'ONDO', 'WLD', 'RENDER', 'ENA', 'JST', 'ARB', 'FIL', 'TRUMP', 'DASH', 'FET']


,BTC,ETH,BNB,XRP,SOL,TRX,DOGE,ADA,BCH,LINK,ZEC,XLM,LTC,AVAX,SUI,TON,SHIB,WLFI,TAO,DOT,UNI,NEAR,ASTER,PEPE,AAVE,ONDO,WLD,RENDER,ENA,JST,ARB,FIL,TRUMP,DASH,FET
timestamp,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-03-02 00:05:00+00:00,+0.1739%,+0.2673%,-0.0355%,+0.0664%,+0.2390%,+0.0000%,+0.1847%,-0.0367%,+0.2034%,+0.2301%,+0.3080%,+0.0000%,+0.1688%,+0.1106%,+0.3035%,+0.0000%,+0.1802%,-0.0923%,+0.3930%,+0.0651%,+0.1332%,+0.4352%,+0.2809%,+0.2890%,+0.0263%,+0.2377%,+0.1299%,+0.0000%,+0.0965%,+0.0627%,+0.4016%,+0.1036%,+0.1745%,+0.1564%,+0.1326%
2026-03-02 00:10:00+00:00,-0.0401%,-0.1438%,-0.2682%,+0.0516%,+0.0119%,+0.0356%,+0.0542%,+0.0367%,-0.3383%,-0.1148%,+0.0465%,-0.0647%,+0.0375%,-0.1105%,-0.3250%,-0.0833%,-0.1799%,+0.2773%,-0.1119%,-0.0651%,+0.1597%,+0.3466%,+0.0000%,+0.0000%,-0.0351%,-0.0395%,-0.1816%,+0.1490%,-0.0964%,-0.1044%,-0.1000%,+0.1035%,-0.2322%,+0.0937%,+0.0662%
2026-03-02 00:15:00+00:00,-0.0955%,+0.0581%,-0.1896%,-0.0074%,+0.0000%,+0.0000%,-0.0542%,+0.0733%,+0.2263%,+0.0000%,-0.0232%,-0.1943%,+0.0000%,-0.1106%,+0.1349%,+0.1667%,+0.0000%,+0.0000%,+0.0000%,+0.0651%,+0.0531%,+0.6908%,-0.2801%,+0.2882%,+0.1141%,-0.0791%,+0.1299%,-0.1488%,+0.1931%,-0.0209%,+0.2002%,+0.2068%,+0.1746%,+0.0000%,-0.0662%
2026-03-02 00:20:00+00:00,+0.1397%,+0.2041%,+0.0763%,+0.0884%,+0.1669%,-0.0356%,+0.1735%,+0.2564%,+0.0452%,+0.0000%,+0.2977%,+0.0649%,-0.0749%,+0.0000%,-0.0112%,+0.1664%,+0.5405%,-0.3687%,-0.1120%,+0.5205%,+0.2921%,-0.0858%,+0.0000%,+0.2874%,+0.3770%,+0.1979%,+0.0779%,+0.5961%,-0.1927%,+0.2509%,+0.2997%,+0.1032%,+0.0871%,+0.4056%,+0.1987%
2026-03-02 00:25:00+00:00,-0.1192%,-0.1303%,-0.1557%,-0.0736%,-0.1547%,-0.0356%,-0.1515%,-0.0365%,-0.0226%,-0.2299%,+0.0325%,+0.0000%,-0.1124%,-0.1107%,-0.2022%,+0.0831%,-0.1792%,+0.1850%,-0.2242%,+0.0647%,-0.2912%,+0.1717%,-0.4213%,-0.2865%,-0.1922%,+0.0000%,-0.1297%,+0.0000%,+0.0965%,+0.0209%,+0.0000%,-0.1031%,-0.1161%,-0.0932%,-0.2644%


In [46]:
# --- Visualisierung 4: Korrelations-Heatmap aller Assets ---
corr_matrix = pd.DataFrame(pc, columns=syms).corr()

fig, ax = plt.subplots(figsize=(max(8, len(syms) * 0.35), max(6, len(syms) * 0.3)))
im = ax.imshow(corr_matrix.values, vmin=-1, vmax=1, cmap="RdYlGn", aspect="auto")
plt.colorbar(im, ax=ax, label="Pearson ρ")
ax.set_xticks(range(len(syms))); ax.set_xticklabels(syms, rotation=90, fontsize=7)
ax.set_yticks(range(len(syms))); ax.set_yticklabels(syms, fontsize=7)
ax.set_title(f"Paarweise Korrelation der Point-Cloud-Dimensionen\n{WINDOW_DAYS}-Tage-Fenster · {result['n_points']} Punkte")
plt.tight_layout()
plt.savefig("viz4_heatmap.png", dpi=120)
plt.show()
plt.close("all")

pairs = (corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)).stack().sort_values(ascending=False))
print("Top-10 korrelierende Paare:")
print(pairs.head(10).to_string())
print("\nTop-5 anti-korrelierende Paare:")
print(pairs.tail(5).to_string())

Top-10 korrelierende Paare:
BTC   ETH     0.901905
ETH   BNB     0.890939
      SOL     0.889852
BTC   BNB     0.885473
      SOL     0.878576
BNB   SOL     0.875909
ETH   LINK    0.869733
DOGE  ADA     0.857440
ADA   SUI     0.852531
SOL   LINK    0.838215

Top-5 anti-korrelierende Paare:
JST    FET      0.146601
ASTER  JST      0.146315
TAO    JST      0.136740
TRX    JST      0.128103
JST    TRUMP    0.116786


C:\Users\ilija\AppData\Local\Temp\ipykernel_39248\1056298658.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [47]:
# --- Visualisierung 6: Interaktive 3D Point Cloud (frei wählbare Achsen) ---
# X/Y/Z-Achse per Dropdown beliebig auf ein Asset legen -> "durch" die
# n-dimensionale Wolke wandern. Farbe = Return auf der X-Achse (diverging).
from cloudvisualisation import plot_point_cloud_3d_interactive

plot_point_cloud_3d_interactive(pc, syms, timestamps=result["timestamps"])


interactive(children=(Dropdown(description='X-Achse:', options=('BTC', 'ETH', 'BNB', 'XRP', 'SOL', 'TRX', 'DOG…

## Vietoris-Rips-Filtration: Entstehen und Verschwinden von Clustern

Gemaess der PDF-Klaerung ("Additional Information for Bachelor Thesis May 2026"):

- Bei H0 (zusammenhaengende Komponenten) sind **alle** Cluster bei `epsilon=0` geboren
  (jedes Asset ist zunaechst isoliert).
- Die Persistenz eines Clusters wird daher vollstaendig durch seinen **Death-Scale**
  bestimmt: den `epsilon`-Wert, bei dem er mit einem anderen Cluster verschmilzt.
- `scipy.cluster.hierarchy.linkage(method='single')` auf der Korrelationsdistanzmatrix
  liefert exakt diese Merge-Hoehen — identisch zur H0-Filtration des Vietoris-Rips-Komplexes.

Zwei Visualisierungen dazu:
1. **Epsilon-Slider**: Assets als Knoten (MDS-Layout), Kanten erscheinen bei Distanz <= epsilon.
2. **H0-Persistenz-Barcode**: ein Balken pro Merge-Event von birth=0 bis death=epsilon.

In [48]:
# --- Korrelationsdistanzmatrix aus der Point Cloud ---
# pc ist die Return-Matrix (n_timestamps x n_assets); helper.correlation_distance
# erwartet ein DataFrame mit Assets als Spalten (genau wie "returns" in build_window_input).
returns_df = pd.DataFrame(pc, columns=syms, index=result["timestamps"])
dist_matrix = helper.correlation_distance(returns_df)

print(f"Distanzmatrix-Shape: {dist_matrix.shape}")
print(f"epsilon-Bereich: [0, {dist_matrix.max():.4f}]  (sqrt(2) = {np.sqrt(2):.4f} = keine Korrelation)")

Distanzmatrix-Shape: (35, 35)
epsilon-Bereich: [0, 1.3291]  (sqrt(2) = 1.4142 = keine Korrelation)


In [ ]:
# --- Visualisierung 7: Epsilon-Slider (Vietoris-Rips 1-Skelett) ---
# Assets als Knoten (MDS-Layout aus der Distanzmatrix). Kanten erscheinen, sobald
# die paarweise Korrelationsdistanz <= epsilon ist. Beim Hochziehen des Sliders sieht
# man live, wie isolierte Assets (epsilon=0) zu Cliquen verschmelzen.
from cloudvisualisation import plot_vietoris_rips_epsilon_slider

plot_vietoris_rips_epsilon_slider(dist_matrix, syms)

interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='epsilon:', layout=Layout(wi…

In [50]:
# --- Visualisierung 8: H0-Persistenz-Barcode ---
# Ein Balken pro Merge-Event (birth=0 -> death=epsilon). Lange Balken mit vielen
# beteiligten Assets = strukturell stabile Cliquen (hohe Persistenz = robust gegen
# Marktvolatilitaet, siehe PDF "Another important point").
from cloudvisualisation import plot_h0_persistence_barcode

z_linkage = plot_h0_persistence_barcode(dist_matrix, syms)

print(f"\nGroesste Merge-Events (hoechste Persistenz = death, da birth immer 0):")
merge_df = pd.DataFrame(z_linkage, columns=["cluster_1", "cluster_2", "death", "size"])
print(merge_df.sort_values("death", ascending=False).head(5).to_string(index=False))


Groesste Merge-Events (hoechste Persistenz = death, da birth immer 0):
 cluster_1  cluster_2    death  size
      29.0       67.0 1.216978  35.0
       5.0       66.0 1.189314  34.0
      32.0       65.0 1.048745  33.0
      22.0       64.0 0.994784  32.0
      15.0       63.0 0.992272  31.0
